# Day 64 — Retail Demand Predictor Mini-Project

This notebook covers Retail Demand Forecasting Blueprint, Tabular Schema Design, Synthetic Economic Simulation, Fuel/Transport Cost Elasticity, and Seasonal Demand Multipliers in Nigerian Markets

## Validation goals
- Design a comprehensive project blueprint and tabular schema for retail demand prediction tailored to emerging market dynamics
- Implement realistic synthetic data generation using Pandas and NumPy encoding economic relationships like price elasticity and logistics overhead
- Analyze why localized factors such as fuel/transport costs (`fuel_transport_cost_ngn`) and seasons are critical drivers alongside base retail prices
- Establish strict separation between input features ($X$) and target labels ($Y$) to prevent look-ahead bias and data leakage

### 1. Project Blueprint: Nigeria Retail Demand Predictor

In retail and supply chain management, predicting consumer demand accurately prevents stockouts during peak seasons and avoids costly overstocking. In emerging and dynamic markets like Nigeria, retail demand is heavily influenced by unique localized factors—such as fluctuating transport/PMS fuel costs across states, Value Added Tax (VAT) adjustments, seasonal festive periods (like December holiday rushes and Eid celebrations), and regional purchasing power.

### Project Specs

Target Label ($Y$): demand_units (The exact quantity of an item expected to be sold in a given week).
- Core Inputs / Features ($X$):
    - historical_sales: Past average weekly sales volume.
    - price: Unit selling price in Naira (₦).
    - vat: Applicable Value Added Tax rate (e.g., standard 7.5%).
    - category: Product category (e.g., Groceries, Electronics, FMCG, Apparel).
    - location: Commercial hub / state (e.g., Lagos - Alaba/Balogun, Kano - Kurmi, Abuja - Wuse, Rivers - Port Harcourt).
    - season: Environmental/calendar phase (e.g., Dry Season, Rainy Season, Harmattan, Festive Peak).
    - fuel_transport_cost: Average inter-state or intra-city logistics/transport cost per unit, which directly impacts retail pricing and consumer demand elasticity.

### 2. Designing the Dataset Schema

Before writing any modeling code, an ML engineer must carefully define the schema. For our project, we will construct a tabular dataset where each row represents a weekly product-store observation.

| Column Name | Data Type | Description |
| --- | --- | --- |
| `product_id` | Categorical / String | Unique identifier for the retail SKU |
| `location` | Categorical | Commercial hub (Lagos, Kano, Abuja, Port Harcourt, Onitsha) |
| `category` | Categorical | Product sector (Groceries, FMCG, Apparel, Electronics) |
| `price_ngn` | Continuous (Float) | Unit selling price in Nigerian Naira (₦) |
| `vat_rate` | Continuous (Float) | Active VAT percentage (e.g., 0.075) |
| `fuel_transport_cost_ngn` | Continuous (Float) | Logistics and fuel overhead per shipment batch |
| `season` | Categorical | Seasonal marker (Regular, Rainy, Harmattan, Festive) |
| `historical_sales_lag` | Continuous (Float) | Units sold in the previous matching period |
| `demand_units` | Continuous (Float) | Target Label ($Y$): Actual units demanded/sold |

### 3. Generating a Realistic Synthetic Dataset in Python

Because real-world proprietary retail logs are rarely public, data scientists frequently build robust synthetic simulators that encode realistic economic relationships (e.g., higher fuel transport costs and higher prices decrease demand; festive seasons and lower prices increase demand).

Let's write a clean Python script using Pandas and NumPy to initialize our project dataset.

In [1]:
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)

n_samples = 2500

# 1. Generate categorical features
locations = ['Lagos', 'Kano', 'Abuja', 'Port Harcourt', 'Onitsha']
categories = ['Groceries', 'FMCG', 'Apparel', 'Electronics']
seasons = ['Regular', 'Rainy', 'Harmattan', 'Festive']

location_data = np.random.choice(locations, size=n_samples)
category_data = np.random.choice(categories, size=n_samples)
season_data = np.random.choice(seasons, size=n_samples, p=[0.5, 0.2, 0.1, 0.2])

# 2. Generate numerical features with realistic distributions
prices = np.random.uniform(500, 75000, size=n_samples)         # Unit price in NGN
vat_rates = np.full(n_samples, 0.075)                           # Standard 7.5% VAT
fuel_transport_costs = np.random.uniform(1500, 15000, size=n_samples) # Logistics overhead
historical_sales = np.random.randint(20, 500, size=n_samples).astype(float)

# 3. Simulate target demand using underlying economic rules + random noise
# Rule: Higher price & transport cost reduce demand; festive season and historical sales boost demand
base_demand = (
    0.6 * historical_sales 
    - 0.002 * prices 
    - 0.005 * fuel_transport_costs
)

# Apply multipliers for seasons
season_multipliers = {'Regular': 1.0, 'Rainy': 0.9, 'Harmattan': 0.95, 'Festive': 1.4}
season_effect = np.array([season_multipliers[s] for s in season_data])

# Calculate final demand with some Gaussian noise
demand_units = (base_demand * season_effect) + np.random.normal(0, 15, size=n_samples)
demand_units = np.clip(demand_units, 5, 2000).astype(int) # Ensure no negative demand

# 4. Assemble into a pristine Pandas DataFrame
df_retail = pd.DataFrame({
    'location': location_data,
    'category': category_data,
    'season': season_data,
    'price_ngn': prices.round(2),
    'vat_rate': vat_rates,
    'fuel_transport_cost_ngn': fuel_transport_costs.round(2),
    'historical_sales_lag': historical_sales,
    'demand_units': demand_units
})

df_retail.to_csv("retail_demand_predictor.csv", index=False)


print("--- Nigeria Retail Demand Dataset Initialized ---")
print(df_retail.head(5))
print(f"\nDataset Shape: {df_retail.shape}")

--- Nigeria Retail Demand Dataset Initialized ---
        location     category   season  price_ngn  vat_rate  \
0  Port Harcourt    Groceries  Regular   69317.99     0.075   
1        Onitsha  Electronics    Rainy   61699.61     0.075   
2          Abuja    Groceries    Rainy   26877.81     0.075   
3        Onitsha  Electronics  Regular   49358.65     0.075   
4        Onitsha    Groceries  Regular   40527.04     0.075   

   fuel_transport_cost_ngn  historical_sales_lag  demand_units  
0                  8589.40                 471.0           109  
1                  4240.41                  90.0             5  
2                  7600.23                 156.0             7  
3                 12258.79                 163.0             5  
4                  9470.75                 408.0           125  

Dataset Shape: (2500, 8)


### Guided Question:

In the context of retail demand forecasting in Nigeria, why is including features like fuel_transport_cost_ngn and season just as important as the product's base price?

### Answer:

In Nigeria's retail environment, relying solely on a product’s base price (`price_ngn`) misses the structural and macroeconomic shocks that dictate whether goods actually reach the shelf and whether consumers can afford them. Including `fuel_transport_cost_ngn` and season captures supply-side bottlenecks, real purchasing power erosion, and sharp cultural/climatic demand shifts.

**1. Why fuel_transport_cost_ngn is `Critical`**

In Nigeria, road transport is the primary mode of moving freight between commercial hubs (such as moving goods from ports in Lagos/Port Harcourt to Kano, Abuja, or Onitsha), and diesel/petrol fluctuations ripple immediately across the entire retail value chain.
- Pass-Through Inflation & Margin Compression: Retailers frequently pass surging logistics and generator fuel expenses directly onto consumers. Even if a product's base factory price stays flat, higher transport overhead drives up the final landed shelf price, suppressing unit demand—especially for price-elastic FMCG and grocery items
- Erosion of Consumer Disposable Income: High fuel and transit costs don't just affect retailers; they directly raise commuter fares and household energy costs. When consumers spend a larger share of their income on daily transport and cooking/generator fuel, discretionary spending on categories like Apparel and Electronics drops sharply, even if the retail prices of those items haven't changed.
- Supply Chain Friction & Stockouts: Spikes in fuel prices or fuel scarcity can delay shipments or make restocking low-margin goods unprofitable for distributors. A model without transport cost data might mistake a drop in recorded sales (demand_units) for low consumer interest, when it was actually caused by a logistics-induced stockout. 

**2. Why season is Critical**

Retail demand in Nigeria does not follow a smooth curve; it is heavily punctuated by distinct climatic and cultural periods that invert normal price-demand relationships.   
- Breaking Standard Price Elasticity (Festive): During major cultural and religious periods (December/Christmas, Eid-el-Fitr, Eid-el-Kabir), demand surges simultaneously with rising prices. Without a season indicator, a model learning from historical data might conclude that higher prices cause higher sales, severely corrupting its price elasticity estimates.
- Climatic Shifts in Category Demand (Rainy vs. Harmattan):
    - Rainy Season: Heavy flooding and degraded road conditions worsen transit times and spoilage for agricultural groceries, while shifting apparel demand toward rain gear and reducing foot traffic to open-air markets.
    - Harmattan: The dry, dusty, and cold weather between November and February triggers predictable spikes in specific categories—such as skincare/lotions (FMCG), lip balms, warm clothing (Apparel), and air purifiers/humidifiers (Electronics)—while dampening demand for cold beverages.
- Inventory Pre-Positioning: Because transit times lengthen during peak rainy months (due to bad roads) and festive months (due to severe highway congestion), knowing the season helps distinguish between organic demand shifts and fulfillment delays. 

**Summary of Interaction**

In a predictive model, `price_ngn` tells you the sticker price of the item, while `fuel_transport_cost_ngn` acts as a proxy for macroeconomic pressure and supply-chain friction, and season acts as a behavioral multiplier that explains why consumers buy differently at different times of the year.

### Guided Question:

In our project schema, why is `demand_units` designated as our target label ($Y$) rather than one of the input features ($X$)?

### Answer:

In supervised machine learning, designating demand_units as the target label ($Y$) rather than an input feature ($X$) comes down to temporal logic, causality, and the fundamental purpose of forecasting.

**1. The Trap of "Look-Ahead Bias" (Data Leakage)**

In predictive modeling, $Y$ represents the future outcome you are trying to estimate, while $X$ represents the information you realistically have access to at the time the prediction is made.
- `demand_units` measures the actual units sold during the target period.
- If you included actual sales as an input feature ($X$) to predict those same sales, your model would be "cheating." It would rely on knowing the exact answer in advance, rendering it completely useless for predicting future or unseen periods.

**2. Inputs vs. Behavioral Outcomes**

The features in your $X$ matrix (`price_ngn, fuel_transport_cost_ngn, season, location, category`) represent the environment and economic conditions under which retail decisions happen:
- They are conditions you can observe, set, or forecast ahead of time (e.g., you know the fuel transport cost today, you know what season it is, and you set the base price).
- `demand_units` is the consumer's response to those conditions. You cannot feed the consumer's reaction into the model as an input because the reaction is precisely what you are trying to calculate.

**3. The Distinction Between Lagged Sales and Target Demand**

It is easy to confuse `demand_units` with `historical_sales_lag`, but they serve entirely different roles in the pipeline:
- historical_sales_lag ($X$): Represents sales from a past matching period (e.g., sales from last month or the same week last year). Because that time has already passed, that data is fully known and serves as a powerful baseline predictor.
- `demand_units` ($Y$): Represents the target sales volume of the current or future horizon that the algorithm must learn to map the $X$ features to.
**4. Real-World Business Value**

Retailers build demand forecasting models weeks or months in advance to optimize inventory purchasing, manage warehouse space in hubs like Kano or Lagos, and allocate working capital. A forecasting model must take known environmental variables ($X$) and project the unknown demand ($Y$) so the business can plan ahead of time.

### Guided Question:

In the synthetic data generation snippet above, why did we use `np.clip()` on the final `demand_units` array?

### Answer:

In the synthetic data generation snippet, `np.numpy.clip()` is used to prevent mathematically possible but economically impossible values from corrupting the dataset.

The combination of linear formulas and random noise can easily generate unrealistic data points. Specifically, clipping serves three critical functions:

**1. Eliminating Negative Demand**

The formula calculates base_demand by subtracting weighted costs (price and logistics) from historical sales, and then adds Gaussian noise (`np.random.normal(0, 15)`):

`demand_units = (base_demand * season_effect) + np.random.normal(0, 15, size=n_samples)`

If a sample randomly selects a very high price_ngn and `fuel_transport_cost_ngn` alongside a low `historical_sales`, the raw calculation can yield a negative number (e.g., `-12` units sold). Because a retail store cannot sell negative units of an item, clipping at a minimum of `5` forces these edge cases into a realistic baseline.

**2. Enforcing Operational Boundaries (Floor and Ceiling)**

By calling `np.clip(demand_units, 5, 2000)`, you establish strict operational guardrails:

- The Floor (`5`): Ensures every retail SKU has a baseline level of baseline activity, avoiding zeros or negative numbers that can break certain log-transformations or loss functions (like MAPE or Poisson regression).
- The Ceiling (`2000`): Acts as a safeguard against extreme random noise spikes that might produce unrealistic outlier demand values (e.g., `50,000` units sold for a standard `SKU`).

**3. Clean Data Type Conversion**

Retail demand is counted in discrete units (`integers`), not continuous fractions. Clipping the array keeps the values in a stable numeric range before casting them cleanly to integers with `.astype(int)`.